# Count

## Imports

In [ ]:
import json
import subprocess
import sys
import threading
import time
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn


## Config

In [ ]:
_cwd = Path.cwd()
ROOT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent
CKPT = ROOT / 'models' / 'digits_mlp.pth'
AUDIO_DIR = ROOT / 'audio' / 'uzbek'

SOURCE = 0
MIRROR = True
STABLE_FRAMES = 10
LOST_FRAMES = 8
CONF_THRESH = 0.55
MUTE = False

UZBEK = {0:'nol', 1:'bir', 2:'ikki', 3:'uch', 4:"to'rt",
         5:'besh', 6:'olti', 7:'yetti', 8:'sakkiz', 9:"to'qqiz"}


## Model

In [ ]:
class HandMLP(nn.Module):
    def __init__(self, num_classes, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, hidden), nn.ReLU(),
            nn.BatchNorm1d(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.BatchNorm1d(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)

def normalize_hand(landmarks):
    if landmarks.shape != (21, 3):
        return None
    if np.abs(landmarks).sum() < 1e-6:
        return None
    wrist = landmarks[0]
    mid_mcp = landmarks[9]
    palm = float(np.linalg.norm(mid_mcp - wrist))
    if palm < 0.02 or palm > 0.5:
        return None
    centered = landmarks - wrist
    return (centered / palm).reshape(-1).astype(np.float32)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    m = HandMLP(ckpt['num_classes'], hidden=ckpt.get('hidden', 128), dropout=ckpt.get('dropout', 0.2))
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    labels = {int(k): v for k, v in ckpt['labels'].items()}
    return m, ckpt['mean'], ckpt['std'], labels


## Audio

In [ ]:
class Speaker:
    AUDIO_EXTS = ('mp3', 'm4a', 'wav', 'ogg', 'aac')

    def __init__(self, audio_dir):
        self.audio_dir = audio_dir
        self._proc = None
        self._lock = threading.Lock()
        self._files = {}
        self._alias_counter = 0
        self._last_alias = None
        if audio_dir.exists():
            for ext in self.AUDIO_EXTS:
                for p in audio_dir.glob(f'*.{ext}'):
                    try:
                        d = int(p.stem)
                        if 0 <= d <= 9 and d not in self._files:
                            self._files[d] = p
                    except ValueError:
                        continue
        self.available = bool(self._files)

    def _play(self, path):
        if sys.platform == 'darwin':
            return subprocess.Popen(
                ['afplay', str(path)],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        if sys.platform == 'win32':
            import ctypes
            winmm = ctypes.windll.winmm
            self._alias_counter += 1
            alias = f'sl_snd_{self._alias_counter}'
            if self._last_alias:
                winmm.mciSendStringW(f'close {self._last_alias}', None, 0, 0)
            winmm.mciSendStringW(f'open "{path}" alias {alias}', None, 0, 0)
            winmm.mciSendStringW(f'play {alias}', None, 0, 0)
            self._last_alias = alias
            return None
        for player, extra in (('ffplay', ['-nodisp', '-autoexit', '-loglevel', 'quiet']),
                              ('paplay', []), ('aplay', ['-q'])):
            try:
                return subprocess.Popen(
                    [player] + extra + [str(path)],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
            except FileNotFoundError:
                continue
        return None

    def speak(self, digit):
        if not self.available or digit not in self._files:
            return
        path = self._files[digit]
        with self._lock:
            if self._proc and self._proc.poll() is None:
                try:
                    self._proc.terminate()
                except Exception:
                    pass
            try:
                self._proc = self._play(path)
            except FileNotFoundError:
                self.available = False
            except Exception:
                pass


## Drawing

In [ ]:
CYAN = (255, 220, 0)
MAGENTA = (230, 80, 255)
LOCK = (120, 255, 80)
PENDING = (0, 180, 255)
WHITE = (240, 240, 240)
DIM = (120, 120, 120)
RED = (80, 80, 240)

def neon(frame, text, org, scale, color, thickness=2, font=cv2.FONT_HERSHEY_DUPLEX):
    dim = tuple(int(c * 0.25) for c in color)
    cv2.putText(frame, text, org, font, scale, dim, thickness + 5, cv2.LINE_AA)
    cv2.putText(frame, text, org, font, scale, color, thickness, cv2.LINE_AA)

def panel(frame, x1, y1, x2, y2, alpha=0.6, color=CYAN):
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (12, 8, 4), -1)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    L = 18
    for cx, cy, dx, dy in [(x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)]:
        cv2.line(frame, (cx, cy), (cx + L * dx, cy), color, 2, cv2.LINE_AA)
        cv2.line(frame, (cx, cy), (cx, cy + L * dy), color, 2, cv2.LINE_AA)

def hand_brackets(frame, landmarks, w, h, color):
    xs = [int(p.x * w) for p in landmarks]
    ys = [int(p.y * h) for p in landmarks]
    pad = 28
    x1 = max(min(xs) - pad, 0)
    y1 = max(min(ys) - pad, 0)
    x2 = min(max(xs) + pad, w - 1)
    y2 = min(max(ys) + pad, h - 1)
    L = 22
    for cx, cy, dx, dy in [(x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)]:
        cv2.line(frame, (cx, cy), (cx + L * dx, cy), color, 2, cv2.LINE_AA)
        cv2.line(frame, (cx, cy), (cx, cy + L * dy), color, 2, cv2.LINE_AA)

def hand_skeleton(frame, landmarks, connections, w, h, color):
    for a, b in connections:
        p1 = (int(landmarks[a].x * w), int(landmarks[a].y * h))
        p2 = (int(landmarks[b].x * w), int(landmarks[b].y * h))
        cv2.line(frame, p1, p2, color, 1, cv2.LINE_AA)
    for lm in landmarks:
        cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, color, -1, cv2.LINE_AA)

def arc_progress(frame, center, radius, frac, color):
    cv2.ellipse(frame, center, (radius, radius), -90, 0, 360, (55, 55, 55), 3, cv2.LINE_AA)
    if frac > 0:
        cv2.ellipse(frame, center, (radius, radius), -90, 0, int(frac * 360), color, 3, cv2.LINE_AA)

def stop_button(frame, x1, y1, x2, y2, hovered=False):
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (10, 6, 20), -1)
    alpha = 0.85 if hovered else 0.7
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    color = RED
    thick = 2 if hovered else 1
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick, cv2.LINE_AA)
    L = 12
    for cx, cy, dx, dy in [(x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)]:
        cv2.line(frame, (cx, cy), (cx + L * dx, cy), color, 2, cv2.LINE_AA)
        cv2.line(frame, (cx, cy), (cx, cy + L * dy), color, 2, cv2.LINE_AA)
    cv2.circle(frame, (x1 + 18, (y1 + y2) // 2), 4, color, -1, cv2.LINE_AA)
    text = 'STOP'
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, 0.65, 2)
    tx = x1 + 30 + ((x2 - x1 - 30) - tw) // 2
    ty = y1 + (y2 - y1 + th) // 2 - 2
    neon(frame, text, (tx, ty), 0.65, color, 2)

## Load

In [ ]:
model, mean, std, labels = load_checkpoint(CKPT)
name_to_digit = {'zero':0,'one':1,'two':2,'three':3,'four':4,
                 'five':5,'six':6,'seven':7,'eight':8,'nine':9}
idx_to_digit = {i: name_to_digit.get(n, -1) for i, n in labels.items()}
speaker = Speaker(AUDIO_DIR)
print(f'Audio: {"ENABLED" if speaker.available else "DISABLED"} ({AUDIO_DIR})')


## Run

In [ ]:
cap = cv2.VideoCapture(SOURCE)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

mp_hands = mp.solutions.hands
WIN = 'SIGN LANG // 0-9 // UZ'

current_stable = None
stable_counter = 0
no_hand_counter = LOST_FRAMES
last_spoken = None
fps_t = time.time()
fps_n = 0
fps = 0.0

ui = {'stop': False, 'hover': False, 'btn': None}

def on_mouse(event, x, y, flags, param):
    b = ui['btn']
    if b is None:
        return
    ui['hover'] = b[0] <= x <= b[2] and b[1] <= y <= b[3]
    if event == cv2.EVENT_LBUTTONDOWN and ui['hover']:
        ui['stop'] = True

cv2.namedWindow(WIN)
cv2.setMouseCallback(WIN, on_mouse)

with mp_hands.Hands(static_image_mode=False, max_num_hands=1, model_complexity=1,
                    min_detection_confidence=0.6, min_tracking_confidence=0.5) as hands:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if isinstance(SOURCE, int) and MIRROR:
            frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res = hands.process(rgb)
        frame.flags.writeable = True

        h, w = frame.shape[:2]
        digit_pred = None
        conf = 0.0
        hand_lm = None

        if res.multi_hand_landmarks and res.multi_handedness:
            h_lm = res.multi_hand_landmarks[0]
            hand_lm = h_lm.landmark
            hd = res.multi_handedness[0].classification[0].label
            pts = np.array([[p.x, p.y, p.z] for p in h_lm.landmark], dtype=np.float32)
            if hd == 'Left':
                pts = pts.copy()
                pts[:, 0] = -pts[:, 0]
            feat = normalize_hand(pts)
            if feat is not None:
                xn = (feat - mean) / std
                with torch.no_grad():
                    logits = model(torch.from_numpy(xn).float().unsqueeze(0))
                    probs = torch.softmax(logits, 1)[0].numpy()
                i = int(probs.argmax())
                if probs[i] >= CONF_THRESH:
                    digit_pred = idx_to_digit.get(i, None)
                    conf = float(probs[i])
            no_hand_counter = 0
        else:
            no_hand_counter += 1
            stable_counter = 0
            current_stable = None
            if no_hand_counter >= LOST_FRAMES:
                last_spoken = None

        if digit_pred is not None:
            if digit_pred == current_stable:
                stable_counter += 1
            else:
                current_stable = digit_pred
                stable_counter = 1
            if stable_counter >= STABLE_FRAMES and current_stable != last_spoken:
                last_spoken = current_stable
                if not MUTE:
                    speaker.speak(current_stable)

        frac = min(stable_counter / STABLE_FRAMES, 1.0) if current_stable is not None else 0.0
        locked = current_stable is not None and current_stable == last_spoken
        active = LOCK if locked else (PENDING if current_stable is not None else DIM)

        if hand_lm is not None:
            skel_color = active if current_stable is not None else CYAN
            hand_skeleton(frame, hand_lm, mp_hands.HAND_CONNECTIONS, w, h, skel_color)
            hand_brackets(frame, hand_lm, w, h, skel_color)

        neon(frame, 'SIGN LANG // 0-9 // UZ', (20, 36), 0.65, CYAN, 1)
        neon(frame, '[ Q ] quit', (20, 58), 0.45, DIM, 1)

        pw, ph = 300, 230
        px, py = w - pw - 20, 20
        panel(frame, px, py, px + pw, py + ph, alpha=0.62, color=CYAN)

        blink = int(time.time() * 2) % 2 == 0
        scan_color = MAGENTA if blink else DIM
        neon(frame, 'SCAN', (px + 18, py + 30), 0.55, scan_color, 1)
        cv2.circle(frame, (px + pw - 28, py + 24), 5, scan_color, -1, cv2.LINE_AA)

        cx, cy = px + 70, py + 130
        arc_progress(frame, (cx, cy), 44, frac, active)

        if current_stable is not None:
            ds = str(current_stable)
            (tw, th), _ = cv2.getTextSize(ds, cv2.FONT_HERSHEY_DUPLEX, 2.0, 3)
            neon(frame, ds, (cx - tw // 2, cy + th // 2 - 4), 2.0, active, 3)
            neon(frame, UZBEK.get(current_stable, '').upper(), (px + 140, py + 110), 0.85, WHITE, 2)
            neon(frame, f'CONF {conf*100:>3.0f}%', (px + 140, py + 138), 0.4, DIM, 1)
            bar_x1, bar_y = px + 140, py + 148
            bar_x2 = px + pw - 20
            cv2.rectangle(frame, (bar_x1, bar_y), (bar_x2, bar_y + 5), (40, 40, 40), -1)
            cv2.rectangle(frame, (bar_x1, bar_y),
                          (bar_x1 + int(conf * (bar_x2 - bar_x1)), bar_y + 5),
                          active, -1)
            status = 'LOCKED' if locked else 'HOLD'
            neon(frame, status, (px + 140, py + 188), 0.55, active, 1)
        else:
            neon(frame, '--', (cx - 16, cy + 10), 1.2, DIM, 2)
            neon(frame, 'NO SIGNAL', (px + 140, py + 128), 0.65, DIM, 1)
            neon(frame, 'show a hand', (px + 140, py + 156), 0.45, DIM, 1)

        if last_spoken is not None:
            ghost = str(last_spoken)
            (tw, th), _ = cv2.getTextSize(ghost, cv2.FONT_HERSHEY_DUPLEX, 5.5, 4)
            neon(frame, ghost, (30, h - 30), 5.5, LOCK, 4)
            neon(frame, UZBEK[last_spoken].upper(), (30 + tw + 18, h - 40), 1.0, WHITE, 2)
            neon(frame, 'LAST', (30, h - 30 - th - 8), 0.4, DIM, 1)

        btn_w, btn_h = 140, 52
        bx2 = w - 20
        by2 = h - 55
        bx1 = bx2 - btn_w
        by1 = by2 - btn_h
        ui['btn'] = (bx1, by1, bx2, by2)
        stop_button(frame, bx1, by1, bx2, by2, hovered=ui['hover'])

        fps_n += 1
        if time.time() - fps_t > 0.5:
            fps = fps_n / (time.time() - fps_t)
            fps_t = time.time()
            fps_n = 0
        neon(frame, f'{fps:>3.0f} FPS', (w - 110, h - 24), 0.45, DIM, 1)

        cv2.imshow(WIN, frame)
        if ui['stop']:
            break
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()